# EIA Electricity Demand — EDA

**Phase 2.** Run after `scripts/validate_raw_data.py` passes.

Notebooks are for *understanding*. If you find a data problem here,
the fix goes in `src/`, not in a cell.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Notebooks run from notebooks/, but our code lives in the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "eia"
REPORT  = PROJECT_ROOT / "data" / "raw" / "_logs" / "validation_report.json"

print("project root:", PROJECT_ROOT)
print("raw dir exists:", RAW_DIR.exists())

project root: /Users/aryankulkarni/Desktop/BE/honours/Mini Project
raw dir exists: True


In [2]:
df_raw = pd.read_parquet(RAW_DIR)

print("shape:", df_raw.shape)
print("\ncolumns:", list(df_raw.columns))
df_raw.head()

shape: (793160, 11)

columns: ['period', 'respondent', 'respondent_name', 'type', 'type_name', 'value', 'value_units', 'timezone', '_ingested_at', '_source_route', 'year']


,period,respondent,respondent_name,type,type_name,value,value_units,timezone,_ingested_at,_source_route,year
0,2019-01-01,AEC,PowerSouth Energy Cooperative,D,Demand,9721.0,megawatthours,Eastern,2026-08-20T03:38:58.977373+00:00,electricity/rto/daily-region-data,2019
1,2019-01-01,AEC,PowerSouth Energy Cooperative,NG,Net generation,9858.0,megawatthours,Eastern,2026-08-20T03:38:58.977373+00:00,electricity/rto/daily-region-data,2019
2,2019-01-01,AECI,"Associated Electric Cooperative, Inc.",D,Demand,77536.0,megawatthours,Eastern,2026-08-20T03:39:35.844556+00:00,electricity/rto/daily-region-data,2019
3,2019-01-01,AECI,"Associated Electric Cooperative, Inc.",DF,Day-ahead demand forecast,78675.0,megawatthours,Eastern,2026-08-20T03:39:35.844556+00:00,electricity/rto/daily-region-data,2019
4,2019-01-01,AECI,"Associated Electric Cooperative, Inc.",NG,Net generation,73690.0,megawatthours,Eastern,2026-08-20T03:39:35.844556+00:00,electricity/rto/daily-region-data,2019


In [3]:
df_raw[df_raw.respondent == "PJM"].head(8)[
    ["period", "respondent", "type", "type_name", "value"]
]

,period,respondent,type,type_name,value
180,2019-01-01,PJM,D,Demand,1871762.0
181,2019-01-01,PJM,DF,Day-ahead demand forecast,2030888.0
182,2019-01-01,PJM,NG,Net generation,1798497.0
183,2019-01-01,PJM,TI,Total interchange,80606.0
472,2019-01-02,PJM,D,Demand,2197818.0
473,2019-01-02,PJM,DF,Day-ahead demand forecast,2126440.0
474,2019-01-02,PJM,NG,Net generation,2091811.0
475,2019-01-02,PJM,TI,Total interchange,88623.0


In [4]:
from src.data.validate import load_modeling_frame, load_modeling_respondents

respondents = load_modeling_respondents(REPORT)
print(f"{len(respondents)} approved regions")

df = load_modeling_frame(
    RAW_DIR,
    respondents=respondents,
    start="2022-01-01",
    wide=True,
)

print("shape:", df.shape)
print("range:", df.period.min().date(), "->", df.period.max().date())
df.head()

51 approved regions
shape: (86164, 6)
range: 2022-01-01 -> 2026-08-19


,period,respondent,demand,demand_forecast,net_generation,interchange
0,2022-01-01,AECI,73004.0,73020.0,67019.0,-5985.0
1,2022-01-02,AECI,98239.0,97624.0,90693.0,-7546.0
2,2022-01-03,AECI,92650.0,92297.0,90421.0,-2229.0
3,2022-01-04,AECI,84859.0,81437.0,80304.0,-4555.0
4,2022-01-05,AECI,84547.0,82183.0,84206.0,-341.0
